# Preprocessor

This notebook takes care of the heavy lifting before the Streamlit app can run.  
Basically we:
1. Load the dense point cloud reconstructed by COLMAP
2. Segment each registered image with SAM and extract CLIP features for every segment
3. Project 3D points into the images so we know which mask each point falls into
4. Average the CLIP vectors across views and save everything to a `.npz` file

After running this notebook once you can launch the interactive app without waiting for inference again.

In [ ]:
import os
import cv2
import torch
import numpy as np
import open3d as o3d
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
from transformers import CLIPProcessor, CLIPModel

## Config

Change the paths below if your folder structure is different.

In [ ]:
IMAGE_DIR = "./src/frames"
COLMAP_TXT_DIR = "./colmap/model"
OUTPUT_FILE = "./semantic_scene.npz"
DENSE_CLOUD_PATH = "./colmap/fused.ply"

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

## Helper: quaternion to rotation matrix

COLMAP stores camera orientations as quaternions (w, x, y, z).  
We need a proper 3x3 rotation matrix for the projection later.

In [ ]:
def qvec2rotmat(qvec):
    return np.array([
        [1 - 2*qvec[2]**2 - 2*qvec[3]**2,
         2*qvec[1]*qvec[2] - 2*qvec[0]*qvec[3],
         2*qvec[3]*qvec[1] + 2*qvec[0]*qvec[2]],
        [2*qvec[1]*qvec[2] + 2*qvec[0]*qvec[3],
         1 - 2*qvec[1]**2 - 2*qvec[3]**2,
         2*qvec[2]*qvec[3] - 2*qvec[0]*qvec[1]],
        [2*qvec[3]*qvec[1] - 2*qvec[0]*qvec[2],
         2*qvec[2]*qvec[3] + 2*qvec[0]*qvec[1],
         1 - 2*qvec[1]**2 - 2*qvec[2]**2]
    ])

## Step 1: Load COLMAP data

First we read the dense point cloud and downsample it with a voxel grid so it doesn't blow up RAM later.  
Then we parse `cameras.txt` (intrinsics) and `images.txt` (extrinsics) to know where each photo was taken from.

In [ ]:
pcd_raw = o3d.io.read_point_cloud(DENSE_CLOUD_PATH)
print(f"Original points: {len(pcd_raw.points)}")

# voxel downsample — 0.5 cm cubes, keeps things manageable
pcd = pcd_raw.voxel_down_sample(voxel_size=0.005)
xyz = np.asarray(pcd.points)
rgb = np.asarray(pcd.colors)
num_points = len(xyz)
print(f"After downsampling: {num_points}")

In [ ]:
# accumulators — we'll average across views at the end
accumulated_features = np.zeros((num_points, 512), dtype=np.float32)
point_mask_hits = np.zeros(num_points, dtype=np.int32)      # how many masks landed on this point
point_total_visible = np.zeros(num_points, dtype=np.int32)  # how many times the point was visible at all

In [ ]:
# parse cameras.txt — just need the intrinsic matrix K
cameras = {}
with open(os.path.join(COLMAP_TXT_DIR, "cameras.txt"), "r") as f:
    for line in f:
        if line.startswith("#"):
            continue
        elems = line.split()
        cam_id = elems[0]
        K = np.eye(3)
        K[0, 0] = K[1, 1] = float(elems[4])  # focal length
        K[0, 2] = float(elems[5])             # cx
        K[1, 2] = float(elems[6])             # cy
        cameras[cam_id] = K

print(f"Loaded {len(cameras)} camera(s)")

In [ ]:
# parse images.txt — every other line has the actual pose data
images_data = []
with open(os.path.join(COLMAP_TXT_DIR, "images.txt"), "r") as f:
    lines = f.readlines()
    for i in range(0, len(lines), 2):
        if lines[i].startswith("#"):
            continue
        elems = lines[i].split()
        qvec = np.array(tuple(map(float, elems[1:5])))
        t = np.array(tuple(map(float, elems[5:8])))
        cam_id = elems[8]
        img_name = elems[9]
        images_data.append({
            "name": img_name,
            "R": qvec2rotmat(qvec),
            "t": t,
            "K": cameras[cam_id]
        })

print(f"Found {len(images_data)} registered images")

## Step 2: Load SAM and CLIP

SAM (`vit_b`) does the segmentation, CLIP gives us a 512-d feature vector for each segment.  
Make sure you have the SAM checkpoint downloaded (`sam_vit_b_01ec64.pth`).

In [ ]:
sam = sam_model_registry["vit_b"](checkpoint="./sam_vit_b_01ec64.pth").to(device)
mask_generator = SamAutomaticMaskGenerator(sam)

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("Models loaded")

## Step 3 - Main loop: segment -> embed -> project

For every image we:
1. Run SAM to get a set of masks
2. Crop each masked region (with a small margin for context) and pass it through CLIP
3. Project all 3D points into the image plane and check which mask they land in
4. Accumulate the CLIP feature onto those points

We also keep track of how many times a point was *visible* in a frame (even if no mask covered it).  
This lets us compute a consistency score afterwards (points that are consistently recognized as an object across views are more reliable).

In [ ]:
for i, img_data in enumerate(images_data):
    img_path = os.path.join(IMAGE_DIR, img_data["name"])
    if not os.path.exists(img_path):
        continue

    print(f"[{i+1}/{len(images_data)}] {img_data['name']}")
    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    H, W, _ = image_rgb.shape
    total_area = H * W

    # --- SAM segmentation ---
    masks = mask_generator.generate(image_rgb)
    index_map = np.full((H, W), -1, dtype=np.int32)
    mask_features_list = []

    valid_mask_idx = 0
    for ann in masks:
        x, y, w, h = [int(v) for v in ann['bbox']]
        mask_area = ann['area']

        # skip masks that are too small (noise) or too big (background)
        if mask_area < 500 or mask_area > (total_area * 0.25):
            continue

        # crop with a bit of margin so CLIP gets some context
        margin = 15
        y1, y2 = max(0, y - margin), min(H, y + h + margin)
        x1, x2 = max(0, x - margin), min(W, x + w + margin)
        cropped = image_rgb[y1:y2, x1:x2]

        inputs_image = clip_processor(images=cropped, return_tensors="pt").to(device)
        with torch.no_grad():
            feats = clip_model.get_image_features(**inputs_image)
            if hasattr(feats, "pooler_output"):
                feats = feats.pooler_output
            feats = feats / feats.norm(dim=-1, keepdim=True)

        mask_features_list.append(feats.cpu().numpy()[0])
        index_map[ann['segmentation']] = valid_mask_idx
        valid_mask_idx += 1

    # --- 3D -> 2D projection ---
    R, t, K = img_data["R"], img_data["t"], img_data["K"]

    xyz_cam = (R @ xyz.T).T + t
    valid_depth = xyz_cam[:, 2] > 0
    xyz_cam_valid = xyz_cam[valid_depth]
    valid_indices = np.where(valid_depth)[0]

    uv_homog = (K @ xyz_cam_valid.T).T
    u = (uv_homog[:, 0] / uv_homog[:, 2]).astype(int)
    v = (uv_homog[:, 1] / uv_homog[:, 2]).astype(int)

    in_screen = (u >= 0) & (u < W) & (v >= 0) & (v < H)
    u_in = u[in_screen]
    v_in = v[in_screen]
    final_point_indices = valid_indices[in_screen]

    # mark these points as visible in this frame
    point_total_visible[final_point_indices] += 1

    if valid_mask_idx > 0:
        mask_features_array = np.array(mask_features_list)

        # look up which mask (if any) each projected point landed in
        hit_mask_indices = index_map[v_in, u_in]
        hit_mask_valid = hit_mask_indices != -1

        points_to_update = final_point_indices[hit_mask_valid]
        features_to_add = mask_features_array[hit_mask_indices[hit_mask_valid]]

        accumulated_features[points_to_update] += features_to_add
        point_mask_hits[points_to_update] += 1

## Step 4: Average features and save

We only keep points that were covered by at least one SAM mask.  
For those we compute the mean CLIP vector (re-normalised) and a *consistency score* = `mask_hits / total_visible`.  
A high consistency means the point was recognized as part of some object in most views it appeared in.

In [ ]:
semantic_mask = point_mask_hits > 0
print(f"Semantic points: {np.sum(semantic_mask)} / {num_points}")

# mean of accumulated vectors
final_features = accumulated_features[semantic_mask] / point_mask_hits[semantic_mask][:, None]
norms = np.linalg.norm(final_features, axis=1, keepdims=True)
final_features = final_features / (norms + 1e-8)

# multi-view consistency
hits = point_mask_hits[semantic_mask]
visible = point_total_visible[semantic_mask]
consistency_scores = hits / visible

final_xyz = xyz[semantic_mask]
final_rgb = rgb[semantic_mask]

In [ ]:
np.savez_compressed(
    OUTPUT_FILE,
    xyz=final_xyz,
    rgb=final_rgb,
    features=final_features,
    consistency=consistency_scores
)
print(f"Saved to {OUTPUT_FILE}")